# Debiasing the Sinkhorn Cost by Point Optimization

The raw entropic OT cost has a non-zero self-cost and, for large $\varepsilon$, behaves like a linear functional of the moving measure.  This notebook illustrates the bias by optimizing the locations of a discrete measure
$$
\alpha_n = \frac1n\sum_{i=1}^n \delta_{x_i}
$$
against a fixed two-Gaussian target density $\beta$.  We compare gradient descent on the raw cost $\operatorname{OT}_\varepsilon(\alpha_n,\beta)$ with gradient descent on the debiased Sinkhorn divergence
$$
S_\varepsilon(\alpha_n,\beta)
= \operatorname{OT}_\varepsilon(\alpha_n,\beta)
-\frac12\operatorname{OT}_\varepsilon(\alpha_n,\alpha_n)
-\frac12\operatorname{OT}_\varepsilon(\beta,\beta).
$$
Only the source points move; the target is shown as smooth red level sets.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.special import logsumexp

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    figure_dir,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

NAME = "sinkhorn-divergence-debiasing"
OUT = figure_dir(NAME)
rng = np.random.default_rng(2029)


## Target density and quadrature

The displayed target is a continuous Gaussian mixture.  For the numerical Sinkhorn objectives, the same density is represented by a fixed deterministic quadrature cloud sampled from the mixture; this keeps the notebook light while preserving the target geometry.


In [ ]:
means = np.array([[-0.72, -0.18], [0.74, 0.22]])
scales = np.array([[0.25, 0.18], [0.30, 0.24]])
weights = np.array([0.44, 0.56])


def sample_target(m=112):
    counts = np.floor(m * weights).astype(int)
    counts[-1] += m - counts.sum()
    parts = []
    for k, count in enumerate(counts):
        part = rng.normal(loc=means[k], scale=scales[k], size=(count, 2))
        parts.append(part)
    Y = np.vstack(parts)
    return Y[rng.permutation(len(Y))]


def target_density_grid(res=180):
    u = np.linspace(-1.55, 1.55, res)
    v = np.linspace(-1.15, 1.20, res)
    U, V = np.meshgrid(u, v)
    Z = np.zeros_like(U)
    for w, mu, sig in zip(weights, means, scales):
        dx = (U - mu[0]) / sig[0]
        dy = (V - mu[1]) / sig[1]
        Z += w * np.exp(-0.5 * (dx**2 + dy**2)) / (2 * np.pi * sig[0] * sig[1])
    return U, V, Z


Y = sample_target()
b = np.full(len(Y), 1 / len(Y))
U, V, Z = target_density_grid()
window = (-1.55, 1.55, -1.15, 1.20)


## Sinkhorn gradients

For the quadratic cost $c(x,y)=\|x-y\|^2/2$, the envelope theorem gives the gradient with respect to a moving source point:
$$
\nabla_{x_i}\operatorname{OT}_\varepsilon(\alpha_n,\beta)
= \sum_j P_{ij}(x_i-y_j).
$$
For the self-cost $\operatorname{OT}_\varepsilon(\alpha_n,\alpha_n)$, the same point appears in both marginals, so both the row and column contributions are included.


In [ ]:
def squared_cost(X, Y):
    diff = X[:, None, :] - Y[None, :, :]
    return 0.5 * np.sum(diff * diff, axis=2)


def sinkhorn_plan(X, Y, eps, a=None, b=None, n_iter=85):
    n, m = len(X), len(Y)
    if a is None:
        a = np.full(n, 1 / n)
    if b is None:
        b = np.full(m, 1 / m)
    loga = np.log(a)
    logb = np.log(b)
    C = squared_cost(X, Y)
    f = np.zeros(n)
    g = np.zeros(m)
    for _ in range(n_iter):
        f = -eps * logsumexp(logb[None, :] + (g[None, :] - C) / eps, axis=1)
        g = -eps * logsumexp(loga[:, None] + (f[:, None] - C) / eps, axis=0)
    logP = loga[:, None] + logb[None, :] + (f[:, None] + g[None, :] - C) / eps
    return np.exp(np.clip(logP, -745, 40))


def cross_velocity(X, eps):
    a = np.full(len(X), 1 / len(X))
    P = sinkhorn_plan(X, Y, eps, a=a, b=b)
    grad = np.einsum("ij,ijd->id", P, X[:, None, :] - Y[None, :, :])
    return grad / a[:, None]


def self_velocity(X, eps):
    a = np.full(len(X), 1 / len(X))
    P = sinkhorn_plan(X, X, eps, a=a, b=a)
    diff = X[:, None, :] - X[None, :, :]
    row_grad = np.einsum("ij,ijd->id", P, diff)
    # The same moving atom also appears as a target location; this derivative
    # has the opposite sign of diff_{ik}=x_i-x_k.
    col_grad = -np.einsum("ik,ikd->kd", P, diff)
    return (row_grad + col_grad) / a[:, None]


def objective_velocity(X, eps, debiased):
    v = cross_velocity(X, eps)
    if debiased:
        v = v - 0.5 * self_velocity(X, eps)
    return v


## Four small optimizations

All panels start from the same compact cloud.  The raw large-$\varepsilon$ objective moves points toward a single barycentric blob, whereas the debiased objective preserves the two target modes.


In [ ]:
def initial_cloud(n_points=34):
    counts = np.floor(n_points * weights).astype(int)
    counts[-1] += n_points - counts.sum()
    parts = []
    for k, count in enumerate(counts):
        theta = np.linspace(0, 2 * np.pi, count, endpoint=False)
        radius = 0.55 + 0.38 * ((np.arange(count) % 3) / 2)
        cloud = np.column_stack([
            means[k, 0] + scales[k, 0] * radius * np.cos(theta),
            means[k, 1] + scales[k, 1] * radius * np.sin(theta),
        ])
        parts.append(cloud)
    X0 = np.vstack(parts)
    X0 += rng.normal(scale=0.018, size=X0.shape)
    return X0[rng.permutation(len(X0))]


X_INIT = initial_cloud()


def adam_optimize(eps, debiased, n_steps=220, lr=0.035):
    X = X_INIT.copy()
    m = np.zeros_like(X)
    v = np.zeros_like(X)
    beta1, beta2 = 0.88, 0.97
    for t in range(1, n_steps + 1):
        grad = objective_velocity(X, eps, debiased)
        # A mild confinement prevents a few atoms from leaving the displayed window.
        grad += 0.010 * X
        m = beta1 * m + (1 - beta1) * grad
        v = beta2 * v + (1 - beta2) * grad * grad
        mh = m / (1 - beta1**t)
        vh = v / (1 - beta2**t)
        X -= lr * mh / (np.sqrt(vh) + 1e-8)
    return X


configs = {
    "raw-small": dict(eps=0.030, debiased=False, n_steps=110, lr=0.018),
    "debiased-small": dict(eps=0.030, debiased=True, n_steps=110, lr=0.018),
    "raw-large": dict(eps=0.70, debiased=False, n_steps=210, lr=0.040),
    "debiased-large": dict(eps=0.70, debiased=True, n_steps=210, lr=0.030),
}
results = {name: adam_optimize(**cfg) for name, cfg in configs.items()}
for name, X in results.items():
    print(f"{name:15s} center=({X[:,0].mean():+.3f},{X[:,1].mean():+.3f})  spread={np.sqrt(np.var(X, axis=0).sum()):.3f}")


## Export panels

Each panel shows the same target contours and one optimized point cloud.  The panel names and mathematical explanation are supplied by LaTeX, not embedded inside the PDF.


In [ ]:
def draw_panel(X, path):
    fig, ax = plt.subplots(figsize=(2.05, 1.68))
    levels = np.linspace(Z.max() * 0.10, Z.max() * 0.86, 6)
    ax.contourf(U, V, Z, levels=levels, colors=[RED], alpha=0.045)
    ax.contour(U, V, Z, levels=levels, colors=RED, linewidths=0.52, alpha=0.58)
    ax.scatter(
        X[:, 0],
        X[:, 1],
        s=DIRAC_MARKER_SIZE * 0.47,
        marker="o",
        color=BLUE,
        edgecolor="none",
        linewidth=0,
        alpha=0.88,
        zorder=4,
    )
    ax.set_xlim(window[0], window[1])
    ax.set_ylim(window[2], window[3])
    ax.set_aspect("equal")
    remove_axes(ax)
    save_pdf(fig, path, pad_inches=0.025)
    plt.close(fig)


for name, X in results.items():
    draw_panel(X, OUT / f"{name}.pdf")
